# <h1 align="center"> Anotação de variantes SNPs utilizando SnpEff

## <h2 align="justify"><span style="color: blue;">Esse script foi desenvolvido no contexto da dissertação de mestrado do aluno: Artur Gabriel Rodrigues Silva, no programa de pós graduação em Genética e Melhoramento de Plantas UFG (2025-2026), sob a orientação da professora Thannya Nascimento Soares, e coorientação de Marco Aurélio Caldas de Pinho Pessoa Filho da Embrapa Recursos Genéticos e Biotecnologia, e Rhewter Nunes da Universidade Estadual de Goiás.</span>

---

## Descrição do script

<h3 align="justify"> A anotação das variantes foi realizada utilizando o software SnpEff para anotação das variantes SNPs. Foi utilizado o arquivo FASTA contendo o genoma de referência para Dipteryx alata, o arquivo gff contendo a anotação funcional do genoma e por fim o arquivo VCF obtido a partir da pipeline de GATK+Freebayes.

O SnpEff fornece vários níveis de anotação, desde os mais simples (por exemplo, qual gene cada variante está afetando) até anotações extremamente complexas (por exemplo, se uma variante não codificante afetará a expressão de um gene). Deve-se observar que, quanto mais complexas são as anotações, mais elas dependem de predições computacionais. Essas predições podem estar incorretas, portanto os resultados do SnpEff (ou de qualquer algoritmo de predição) não devem ser aceitos cegamente; eles precisam ser analisados e validados de forma independente por experimentos laboratoriais correspondentes (wet-lab).

- `Database`
    - Para prodeceder para anotações, o SnpEff requer bancos de dados confiáveis. Atualmente o banco de dados deles possui mais de 20.000 genomas de referência. <b> Em alguns casos como o nosso precisamos criar nosso próprio banco de dados de maneira manual, que será explicado mais a frente. </b>
    
- `Documentação`
    - Consulte o seguinte link para conferir os comandos possíveis: https://pcingola.github.io/SnpEff/snpeff/commandline/ 

- `ATENÇÃO`
    - Versões mais antigas do GATK (v2.x/v3.x) possui em seu VCF o campo  INFO  pode vir com a tag EFF as versões mais atuais do GATK possuem a tag ANN. Caso seu VCF seja feito em uma versão mais antiga você pode utilizar a tag -o gatk para dizer para o SnpEff uitilizar a tag EFF. RECOMENDAÇÃO: É aconselhável gerar NOVAMENTE SEU VCF nas versões 4.x superiores para evitar possíveis conflitos com o SnpEff.
    - `O SnpEff fará anotação das variantes no campo ANN`
- `Dúvida`
    - No SnpEff, cada gene pode ter várias isoformas de transcritos, e o programa pode anotar variantes em todas elas ou apenas no chamado transcrito canônico. Usar apenas o canônico facilita a análise, porque reduz a redundância e dá consistência ao resultado, mas isso também pode ocultar efeitos importantes em isoformas alternativas. E a escolha impacta diretamente na interpretação biológica das variantes: se buscamos uma visão geral e simplificada, o canônico é útil; se queremos explorar possíveis efeitos em splicing alternativo ou isoformas específicas, precisamos considerar todos os transcritos.
- `ANN`
    - O campo ANN possui alguns subcampos muito característicos (sendo no total de 16 campos que podem ou não estarem preenchidos), confira o que diz respeito cada um em: https://pcingola.github.io/SnpEff/snpeff/inputoutput/.
- `Sequence Ontology`
    - o SnpEff usa "SO" porque está preocupado em descrever elementos da sequência (exon, intron,UTR, splice site) e tipos de variantes (missense, nonsense, synonymous, frameshift).
- `Previsão de impacto`
    - HIGH (Variante com alto impacto na proteína, trucamento, perda de função), MODERATE (Variante não disruptiva que pode alterar a eficácia das proteínas), LOW (Assume-se que são inofensivos ou pouco prováveis de mudar o comportamento proteico), MODIFIER (Variantes não codificantes ou variantes que afetam genes não codificantes, ausência de evidência de impacto).
- `Previsão de perda de função e decaimento mediado por nonsense`
    - Para inferir um LOF real é NECESSÁRIO validação laboratorial.
- `Outputs`
    - O SnpEff para além do VCF com as variantes anotadas, ele irá gerar um `snpEff_summart.html` [-stats ...], também é possível gerar algumas estatísticas a nível de gene em formato tabular `snpEff_genes.txt` (separado por tab) [-csvStats ...].


---

### 1. Preparando ambiente para trabalho

#### Criando ambiente conda com java >= 8

In [ ]:
%%bash

conda create -n snpeff_env openjdk=11
conda activate snpeff_env

#### Instalando o SnpEff

In [ ]:
%%bash

conda install -c bioconda snpeff

---

### 2. Criando seu próprio banco SnpEff

- Para construir umb anco de dados SnpEff você precisa:
    - 1. Do `genoma de referência` tipicamente um arquivo FASTA.
    - 2. Os `arquivos de anotação` contendo informação dos genes, transcritos e exons no genoma (tipicamente arquivos GTF, GeneBank, GFF, RefSeq).
    - 3. Sequências de CDS ou Proteínas: são utilizados para checar se o banco de dados é consistente e não tem erros.

#### 2.1 Configurando um novo genoma

Edite o arquivo de configuração (snpEff.config) para criar um novo genoma

In [ ]:
%%bash

vi ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/snpEff.config

Adicione as seguintes linhas dentro do snpEff.config

In [ ]:
%%bash

# Dipteryx alata genome, version 0.9
Embrapa_UFG_Dala_nuclear_0.9.genome : Dipteryx alata Vogel

#### 2.2 Construindo banco de dados a partir de arquivos GFF

O SnpEff espera que seus arquivos estejam em pastas específicas dentro da estrutura de diretórios de instalação, portanto você deve:
- `Mover seu genoma de referência e gff para:`

In [ ]:
%%bash

# Nome da pasta que o software espera estar o genoma e o arquivo GFF
mkdir ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/

# Crie um diretório o gff com o mesmo nome que você colocou no snpEff.config
mkdir ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/Embrapa_UFG_Dala_nuclear_0.9/

# Movendo via soft link o arquivo gff com anotações
ln -s ~/data/projects/baru/annotation/mikado.loci.clean.step2.primaryTranscripts.gff3 ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/Embrapa_UFG_Dala_nuclear_0.9/genes.gff

#Crie um diretório para o genoma de referência
mkdir ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/genomes/

# Movendo via soft link o genoma
ln -s ~/data/projects/baru/reference/Embrapa_UFG_Dala_nuclear_0.9.fa ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/genomes/Embrapa_UFG_Dala_nuclear_0.9.fa

- `Adicionando sequências de proteínas para validação do banco`

In [ ]:
%%bash

# Renomeie para protein.fa na hora de passar prodiretório /data/Embrapa_UFG_Dala_nuclear_0.9/
ln -s ~/data/projects/baru/annotation/mikado.loci.clean.step2.primaryTranscripts.proteins.faa ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/Embrapa_UFG_Dala_nuclear_0.9/protein.fa

- `Criando banco de dados:`

In [ ]:
%%bash

# Se localize no seu diretório de trabalho primeiramente, e então execute:
snpeff build -gff3 -noCheckCds -v Embrapa_UFG_Dala_nuclear_0.9

Outputs: 
- sequence.bin, 
- sequence.Dal_02.bin, 
- sequence.Dal_04.bin, 
- sequence.Dal_06.bin, 
- sequence.Dal_08.bin, 
- sequence.Dal_01.bin, 
- sequence.Dal_03.bin, 
- sequence.Dal_05.bin, 
- sequence.Dal_07.bin, 
- snpEffectPredictor.bin.

---

### 4. Executando SnpEff

`Removendo loci "unplaced"`

In [ ]:
%%bash

cat allsamples.step2.tranche90.PASS.biallelic.snps.maf0.05.sitesonly.vcf | grep -v "^unplaced_" > only_chrom_snps.vcf

Descrição dos argumentos utilizados

- `ann` : formato de anotação mais moderno (Sequence Ontology)
- `-v` : Mostra progresso 
- `-stats` : HTML com gráficos e estatísticas
- `-csvStats` : CSV para análises posteriores
- `-lof` : Detecta perda de função (LOF) e NMD

In [ ]:
%%bash

snpeff ann \
  -v \
  -stats dalata_snp_prunned_stats.html \
  -csvStats dalata_snp_prunned_stats.csv \
  -lof \
  Embrapa_UFG_Dala_nuclear_0.9 \
  only_chrom_prunned_snps.vcf > snps_dalata_prunned_annotated.vcf

---

### 5. Extraindo campos para visualziação

Extraindo os campos CHROM, POS, ANN do VCF para visualização em ambiente R

In [ ]:
bcftools query \
    -f '%CHROM\t%POS\t%INFO/ANN\n' \
    snps_dalata_annotated.vcf.gz \
    | gzip -c > ann_raw.tsv.gz